In [1]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

  Using cached lxml-6.0.2-cp312-cp312-win_amd64.whl.metadata (3.7 kB)
Using cached lxml-6.0.2-cp312-cp312-win_amd64.whl (4.0 MB)
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ---------------------------------------  3.1/3.1 MB 23.1 MB/s eta 0:00:01
   ---------------------------------------  3.1/3.1 MB 23.1 MB/s eta 0:00:01
   ---------------------------------------- 3.1/3.1 MB 6.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install -U ddgs

  Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl (369 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Built-in Tool - DuckDuckGo Search

In [31]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

Top - India - news - today ... info@truescoopnews.com ... Copyright © 2025 TrueScoop News . ... News 24 is your stop for those who are inquisitive enough to keep abreast with the ... Today News Headlines in India : Stay Updated with DLSNews24 ... News 24 is your stop for those who are inquisitive enough to keep abreast with the ... Today News Headlines in India : Stay Updated with DLSNews24 Home News Today ’s Trending News : Top Headlines in India and ... Today ’s Trending News : Top Headlines in India and World Updates (August 25, 2025) Here are the top national and international news updates for today , covering key developments in politics, economy, diplomacy, and global events.


In [5]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


## Built-in Tool - Shell Tool

In [32]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('dir')

print(results)

Executing command:
 dir
 Volume in drive H is Training
 Volume Serial Number is 4ED4-8DE5

 Directory of h:\01_Training\ContentSlides_AgenticAI\Code\LangChain\07_Tools

12-12-2025  19:38    <DIR>          .
12-12-2025  19:37    <DIR>          ..
12-12-2025  19:43            32,244 tools.ipynb
12-12-2025  20:11             9,166 tool_calling.ipynb
               2 File(s)         41,410 bytes
               2 Dir(s)  172,948,594,688 bytes free



h:\01_Training\ContentSlides_AgenticAI\Code\LangChain\langchain_env\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## Custom Tools

In [8]:
from langchain_core.tools import tool

In [9]:
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [ ]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers""" #docstring
    return a*b

In [11]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [ ]:
result = multiply.invoke({"a":3, "b":5})

In [13]:
print(result)

15


In [14]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [15]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## Method 2 - Using StructuredTool

In [17]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [18]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14012\3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_14012\3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [19]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [20]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [21]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


## Toolkit

In [28]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [29]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [30]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
